# Serve Qwen2.5-Coder-7B-Instruct via vLLM + public tunnel (ngrok)

Setup before running:
1. Notebook Settings (right sidebar) -> Accelerator: **GPU T4 x2** (or P100).
2. Notebook Settings -> Internet: **ON** (required for downloading the model and for the tunnel).
3. Add your ngrok authtoken as a Kaggle Secret: Add-ons -> Secrets -> add one named
   `NGROK_AUTHTOKEN` with your token from https://dashboard.ngrok.com/get-started/your-authtoken.
   Switched from Cloudflare Quick Tunnel to ngrok because Cloudflare's free quick
   tunnel has a hard ~100s per-request timeout (524 error) that this 7B model on
   a single T4 cannot beat for longer completions (6000-12000 max_tokens steps
   measured at ~220-440s); ngrok's free tier has no such per-request cap.
4. Run all cells. The last cell prints a public HTTPS URL — that's the OpenAI-compatible endpoint.
5. Keep this session running (Save & Run All / commit, or keep the interactive session open) for as long as you need the endpoint. Kaggle caps a session at ~12h and ~30 GPU-hours/week.
6. Send me the printed URL; I'll point the experiment's LLM client at it.

In [ ]:
# 1) Install vLLM (pinned to a version with good Qwen2.5 support) and pyngrok.
#    Pin transformers explicitly: Kaggle's base image ships a newer transformers
#    whose Qwen2Tokenizer dropped `all_special_tokens_extended`, which vllm==0.7.3's
#    tokenizer-caching code still calls -> crashes with AttributeError on startup.
!pip install -q -U vllm==0.7.3 "transformers==4.48.2" "tokenizers>=0.21,<0.22" pyngrok
# vLLM downgrades protobuf; Kaggle generated code requires runtime_version.
!pip install -q "protobuf>=5.29.5,<6"
print('done installing')
import transformers
print('transformers version:', transformers.__version__)

In [ ]:
# 2) Launch vLLM's OpenAI-compatible server in the background.
#    Float16 checkpoint matches the pilot; shard it across the two T4 GPUs.
#    Removed --enforce-eager: CUDA graphs make per-token generation meaningfully
#    faster, which matters here because the free Cloudflare Quick Tunnel has a
#    hard ~100s per-request timeout (524 if exceeded) that --enforce-eager's
#    slower eager-mode generation was tripping on longer completions (6000-12000
#    max_tokens steps). Graph capture adds some startup time/VRAM; watch for OOM
#    during capture -- if it recurs, lower --gpu-memory-utilization slightly
#    (CUDA graphs need extra buffer memory beyond the KV cache) or reduce
#    --max-model-len before re-adding --enforce-eager as a fallback.
import subprocess, time, os

MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
PORT = 8000

server_log = open('/kaggle/working/vllm_server.log', 'w')
server = subprocess.Popen([
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', MODEL,
    '--tensor-parallel-size', '2',
    '--dtype', 'float16',
    '--max-model-len', '16384',
    '--gpu-memory-utilization', '0.88',
    '--port', str(PORT),
    '--served-model-name', 'qwen2.5-coder-7b-instruct',
], stdout=server_log, stderr=subprocess.STDOUT)

print(f'vLLM server starting (pid={server.pid}), tailing log until ready...')
import re, sys, time
ready = False
for _ in range(180):  # up to ~15 min for model download + load + CUDA graph capture
    time.sleep(5)
    with open('/kaggle/working/vllm_server.log') as f:
        text = f.read()
    if 'Uvicorn running on' in text or 'Application startup complete' in text:
        ready = True
        break
    if server.poll() is not None:
        print('Server process exited early, dumping log:')
        print(text[-4000:])
        break
print('READY' if ready else 'NOT READY YET — check /kaggle/working/vllm_server.log')

In [ ]:
# 3) Sanity-check the local server before exposing it
import requests
r = requests.post(f'http://127.0.0.1:{PORT}/v1/chat/completions', json={
    'model': 'qwen2.5-coder-7b-instruct',
    'messages': [{'role': 'user', 'content': 'Say OK'}],
    'max_tokens': 10,
})
print(r.status_code, r.json())

In [ ]:
# 4) Expose via ngrok (no hard per-request timeout, unlike Cloudflare's free quick
#    tunnel). Reads the authtoken from the Kaggle Secret `NGROK_AUTHTOKEN` -- add
#    it via Add-ons -> Secrets before running this cell.
from kaggle_secrets import UserSecretsClient
from pyngrok import ngrok, conf

authtoken = UserSecretsClient().get_secret("NGROK_AUTHTOKEN")
conf.get_default().auth_token = authtoken
ngrok.set_auth_token(authtoken)

# Long response_timeout so ngrok's own client-side wait matches slow 7B generations;
# ngrok's edge itself does not impose a short hard cap like Cloudflare's quick tunnels do.
tunnel = ngrok.connect(PORT, "http", schemes=["https"])
public_url = tunnel.public_url

print('=' * 70)
print('PUBLIC ENDPOINT (OpenAI-compatible):', public_url + '/v1')
print('served-model-name: qwen2.5-coder-7b-instruct')
print('=' * 70)
print('Send this URL back. It will stop working when this notebook session ends.')

In [ ]:
# 5) Keep the session alive (Kaggle kills the kernel once the last cell finishes).
#    Heartbeat every 5 minutes; shuts down after IDLE_MIN minutes without a finished request
#    (FIRST_MIN before the first one) so an unattended run does not burn the weekly GPU quota.
import time, re, requests
IDLE_MIN, FIRST_MIN = 45, 120
def finished():
    try:
        t = requests.get(f'http://127.0.0.1:{PORT}/metrics', timeout=10).text
        return sum(float(v) for v in re.findall(r'^vllm:request_success_total\{[^}]*\} ([0-9.e+]+)$', t, re.M))
    except Exception:
        return None
last_n, last_t, start, i = finished() or 0, time.time(), time.time(), 0
while True:
    time.sleep(300)
    i += 1
    n = finished()
    if n is not None and n > last_n:
        last_n, last_t = n, time.time()
    limit = IDLE_MIN if last_n else FIRST_MIN
    print(f'heartbeat {i}: still serving, finished requests {last_n:.0f}')
    if time.time() - last_t > limit * 60:
        print(f'idle for {limit} min, shutting down')
        break
ngrok.kill()
server.terminate()
